# Conditional Tools - Context-Based Availability

## Purpose
Learn how to dynamically control tool availability based on runtime context and state. This allows tools to appear or disappear based on user permissions, session state, language preferences, or any other runtime condition.

## Key Concepts
- **is_enabled**: Predicate function that determines if a tool should be available
- **Predicate Function**: Returns boolean (True/False) based on context
- **RunContextWrapper**: Provides context information to the predicate
- **Dynamic Tool Selection**: Tools change availability at runtime

## Installation

In [ ]:
#!pip install openai
#!pip install openai-agents
#!pip install aws-bedrock-token-generator

## Authentication Setup

In [ ]:
model_id = "openai.gpt-5.5"

In [ ]:
from openai import AsyncOpenAI
from agents import (
    set_default_openai_client,
    set_default_openai_api,
    set_tracing_disabled,
)
from aws_bedrock_token_generator import provide_token

client = AsyncOpenAI(
    api_key=provide_token(),
    base_url="https://bedrock-mantle.us-east-1.api.aws/openai/v1",
    project="default"
)

set_default_openai_client(client)
set_default_openai_api("responses")
set_tracing_disabled(True)  # OpenAI-platform tracing can't reach Mantle

## Import Libraries

Import `RunContextWrapper` for context management:

In [ ]:
import asyncio
from agents import Agent, AgentBase, Runner, RunContextWrapper
from pydantic import BaseModel

## Step 1: Define Context and Predicates

Create a context model and predicate functions that determine tool availability.

💡 **Key Point**: Predicates receive context and return `True` (enabled) or `False` (disabled).

In [ ]:
class LanguageContext(BaseModel):
    language_preference: str = "french_or_spanish"

def french_enabled(ctx: RunContextWrapper[LanguageContext], agent: AgentBase) -> bool:
    """Tool is available only when French is preferred."""
    return ctx.context.language_preference == "french"

def spanish_enabled(ctx: RunContextWrapper[LanguageContext], agent: AgentBase) -> bool:
    """Tool is available only when Spanish is preferred."""
    return ctx.context.language_preference == "spanish"

## Step 2: Create Language Specialist Agents

Define agents that will be converted to conditional tools:

In [ ]:
spanish_agent = Agent(
    name="spanish_agent",
    model=model_id,
    instructions="You respond in Spanish. Always reply to the user's question in Spanish.",
)

french_agent = Agent(
    name="french_agent",
    model=model_id,
    instructions="You respond in French. Always reply to the user's question in French.",
)

## Step 3: Create Orchestrator with Conditional Tools

Use `is_enabled` parameter when converting agents to tools:

🔍 **How it works**:
- When `language_preference="spanish"`, only Spanish tool is visible
- When `language_preference="french"`, only French tool is visible
- Agent can only see and use available tools

⚡ **Alternative**: You can also hardcode `is_enabled=True` or `is_enabled=False` for static control.

In [ ]:
orchestrator = Agent(
    name="orchestrator",
    model=model_id,
    instructions=(
        "You are a multilingual assistant. You use the tools given to you to respond to users. "
        "You must call ALL available tools to provide responses in different languages. "
        "You never respond in languages yourself, you always use the provided tools."
    ),
    tools=[
        spanish_agent.as_tool(
            tool_name="respond_spanish",
            tool_description="Respond to the user's question in Spanish",
            is_enabled=spanish_enabled,  # Conditional availability
        ),
        french_agent.as_tool(
            tool_name="respond_french",
            tool_description="Respond to the user's question in French",
            is_enabled=french_enabled,  # Conditional availability
        ),
    ],
)

## Step 4: Run with Context

Create a context and run the agent. The tool availability changes based on context:

🎯 **Result**: Only the Spanish tool will be available, so the response will be in Spanish!

In [ ]:
context = RunContextWrapper(LanguageContext(language_preference="spanish"))
result = await Runner.run(orchestrator, "How are you?", context=context.context)
print(result.final_output)

## 🎉 Congratulations!

You've completed the **Conditional Tools** notebook!